In [1]:
import polars as pl

In [27]:
service_file = pl.read_csv(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/push_vbr/model___ext_1805-prov___national-prd___202603-service.csv"
)
quant_file_all = pl.read_csv(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/push_vbr/quantity_data_ext_1805.csv"
)
quant_file_clean_all = pl.read_csv(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/push_vbr/cleaned_quantity_data_ext_1805.csv"
)

In [ ]:
def present(col):
    return pl.col(col).is_not_null() & (pl.col(col) != 0)

In [28]:
quant_file = quant_file_all.filter(
    (present("ver") | present("val"))
    & (pl.col("month").is_in([202603, 202602, 202601, 202512, 202511, 202510, 202509]))
)
quant_file_clean = quant_file_clean_all.filter(
    (present("ver") | present("val"))
    & (pl.col("month").is_in([202603, 202602, 202601, 202512, 202511, 202510, 202509]))
)

In [29]:
ous_in_service = set(service_file.select(pl.col("ou_id")).unique().to_series().to_list())
ous_in_quant = set(quant_file.select(pl.col("ou")).unique().to_series().to_list())
ous_in_quant_clean = set(quant_file_clean.select(pl.col("ou")).unique().to_series().to_list())

In [30]:
only_in_service = ous_in_service - ous_in_quant
only_in_quant = ous_in_quant - ous_in_service
in_clean_not_service = ous_in_quant_clean - ous_in_service

In [35]:
new_rows = pl.DataFrame(
    {
        "period": [202603] * len(only_in_quant),
        "ou_id": list(only_in_quant),
        "Category of the center": ["pma"] * len(only_in_quant),
        "bool verified": [0] * len(only_in_quant),
        "Risk category": [None] * len(only_in_quant),
        "service": [None] * len(only_in_quant),
        "Taux of validation": [None] * len(only_in_quant),
        "Ecart median": [None] * len(only_in_quant),
    }
)

In [36]:
full_service_file = pl.concat([service_file, new_rows], how="vertical")

In [37]:
full_service_file.write_csv(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/push_vbr/mod___202603-service.csv"
)